## This is modified version of the original Covid-19 tracker

In [1]:
import numpy as np
import pandas as pd
import plotly as py
import plotly.express as px
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
init_notebook_mode(connected=True)

In [2]:
# https://immunizationdata.who.int/global?topic=Provisional-measles-and-rubella-data&location=
df = pd.read_csv('measles_data.csv')


In [3]:
df.shape

(25274, 17)

In [4]:
df.info

<bound method DataFrame.info of       Region   Country ISO3  Year  Month  Measles \nsuspect  \
0        AFR   Algeria  DZA  2012      1                8.0   
1        AFR   Algeria  DZA  2012      2               10.0   
2        AFR   Algeria  DZA  2012      3               17.0   
3        AFR   Algeria  DZA  2012      4                7.0   
4        AFR   Algeria  DZA  2012      5               14.0   
...      ...       ...  ...   ...    ...                ...   
25269    WPR  Viet Nam  VNM  2024     10              379.0   
25270    WPR  Viet Nam  VNM  2024     11              584.0   
25271    WPR  Viet Nam  VNM  2024     12              588.0   
25272    WPR  Viet Nam  VNM  2025      1             1413.0   
25273    WPR  Viet Nam  VNM  2025      2              204.0   

       Measles \nclinical  Measles \nepi-linked  Measles \nlab-confirmed  \
0                     6.0                   0.0                      2.0   
1                    10.0                   0.0            

In [5]:
df.head(5)

,Region,Country,ISO3,Year,Month,Measles \nsuspect,Measles \nclinical,Measles \nepi-linked,Measles \nlab-confirmed,Measles \ntotal,Rubella \nclinical,Rubella \nepi-linked,Rubella \nlab-confirmed,Rubella\nTotal,Discarded,Unnamed: 15,Unnamed: 16
0,AFR,Algeria,DZA,2012,1,8.0,6.0,0.0,2.0,8.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN
1,AFR,Algeria,DZA,2012,2,10.0,10.0,0.0,0.0,10.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN
2,AFR,Algeria,DZA,2012,3,17.0,17.0,0.0,0.0,17.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN
3,AFR,Algeria,DZA,2012,4,7.0,5.0,0.0,0.0,5.0,0.0,0.0,1.0,1.0,2.0,NaN,NaN
4,AFR,Algeria,DZA,2012,5,14.0,11.0,0.0,0.0,11.0,0.0,0.0,3.0,3.0,3.0,NaN,NaN


In [6]:
df.tail(5)

,Region,Country,ISO3,Year,Month,Measles \nsuspect,Measles \nclinical,Measles \nepi-linked,Measles \nlab-confirmed,Measles \ntotal,Rubella \nclinical,Rubella \nepi-linked,Rubella \nlab-confirmed,Rubella\nTotal,Discarded,Unnamed: 15,Unnamed: 16
25269,WPR,Viet Nam,VNM,2024,10,379.0,19.0,56.0,256.0,331.0,5.0,0.0,5.0,10.0,48.0,NaN,NaN
25270,WPR,Viet Nam,VNM,2024,11,584.0,37.0,125.0,347.0,509.0,0.0,0.0,1.0,1.0,75.0,NaN,NaN
25271,WPR,Viet Nam,VNM,2024,12,588.0,56.0,134.0,338.0,528.0,0.0,0.0,1.0,1.0,60.0,NaN,NaN
25272,WPR,Viet Nam,VNM,2025,1,1413.0,1264.0,0.0,124.0,1388.0,NaN,NaN,NaN,NaN,25.0,NaN,NaN
25273,WPR,Viet Nam,VNM,2025,2,204.0,182.0,0.0,20.0,202.0,NaN,NaN,NaN,NaN,2.0,NaN,NaN


In [7]:
df.columns = df.columns.str.replace(' ', '')
df.columns = df.columns.str.replace('\n', '-')
df.columns

Index(['Region', 'Country', 'ISO3', 'Year', 'Month', 'Measles-suspect',
       'Measles-clinical', 'Measles-epi-linked', 'Measles-lab-confirmed',
       'Measles-total', 'Rubella-clinical', 'Rubella-epi-linked',
       'Rubella-lab-confirmed', 'Rubella-Total', 'Discarded', 'Unnamed:15',
       'Unnamed:16'],
      dtype='str')

In [8]:
df['Date'] = df['Year'].astype(str) + '-' + df['Month'].astype(str).str.zfill(2)
df_countries = (df.groupby(['Country','Date'], as_index=False)
                .sum()
                .reset_index()
                .sort_values('Date', ascending = False))
df_countries = df_countries.drop_duplicates(subset=['Country'])
df_countries = df_countries[df_countries['Measles-total']>0]
print('Printing dataframe')
print(df_countries.head(5))

Printing dataframe
       index                           Country     Date Region ISO3  Year  \
19353  19353                            Rwanda  2026-11    AFR  RWA  2026   
17875  17875                              Peru  2026-11    AMR  PER  2026   
12945  12945                           Lebanon  2026-11    EMR  LBN  2026   
6538    6538  Democratic Republic of the Congo  2026-11    AFR  COD  2026   
24281  24281          United States of America  2026-11    AMR  USA  2026   

       Month  Measles-suspect  Measles-clinical  Measles-epi-linked  \
19353     11              8.0               8.0                 0.0   
17875     11             20.0               1.0                 0.0   
12945     11              1.0               1.0                 0.0   
6538      11              3.0               0.0                 3.0   
24281     11              0.0               0.0                 0.0   

       Measles-lab-confirmed  Measles-total  Rubella-clinical  \
19353                    0

In [9]:
# Create the static choropleth
fig = go.Figure(data=go.Choropleth(
    locations = df_countries['Country'],
    locationmode = 'country names',
    z = df_countries['Measles-total'],
    colorscale = 'Reds',
    marker_line_color = 'black',
    marker_line_width = 0.5,
))

In [10]:
fig.update_layout(
    title_text = 'Measles cases, latest reported month (November 17, 2026)',
    title_x = 0.5,
    geo = dict(
        showframe = False,
        showcoastlines = False,
        projection_type = 'equirectangular'
    )
)

In [11]:
# Animated Choropleth map
df_countrydate = df[df['Measles-total']>0]
df_countrydate = df_countrydate.groupby(['Date','Country']).sum().reset_index()
df_countrydate

,Date,Country,Region,ISO3,Year,Month,Measles-suspect,Measles-clinical,Measles-epi-linked,Measles-lab-confirmed,Measles-total,Rubella-clinical,Rubella-epi-linked,Rubella-lab-confirmed,Rubella-Total,Discarded,Unnamed:15,Unnamed:16
0,2012-01,Afghanistan,EMR,AFG,2012,1,498.0,0.0,14.0,360.0,374.0,0.0,0.0,16.0,16.0,124.0,0.0,0.0
1,2012-01,Algeria,AFR,DZA,2012,1,8.0,6.0,0.0,2.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2012-01,Angola,AFR,AGO,2012,1,396.0,228.0,125.0,20.0,373.0,0.0,0.0,4.0,4.0,23.0,0.0,0.0
3,2012-01,Australia,WPR,AUS,2012,1,3.0,0.0,0.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2012-01,Austria,EUR,AUT,2012,1,6.0,0.0,4.0,2.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15455,2026-11,Democratic Republic of the Congo,AFR,COD,2026,11,3.0,0.0,3.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15456,2026-11,Lebanon,EMR,LBN,2026,11,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15457,2026-11,Peru,AMR,PER,2026,11,20.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,19.0,0.0,0.0
15458,2026-11,Rwanda,AFR,RWA,2026,11,8.0,8.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
fig = px.choropleth(df_countrydate,
                    locations = 'Country',
                    locationmode = 'country names',
                    color = 'Measles-total',
                    hover_name = 'Country',
                    animation_frame = 'Date'
                    )

In [13]:
fig.update_layout(
    title_text = 'Global spread of Measles',
    title_x = 0.5,
    geo = dict(
        showframe = False,
        showcoastlines = False,
    )
)
fig.show()